In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from scipy.stats import randint, uniform, loguniform
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/raw/Churn_Modelling.csv').drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df['Gender'] = df['Gender'].map({'Female': 0, 'Male': 1})
df = pd.get_dummies(df, columns=['Geography'], drop_first=True, dtype=int)

X = df.drop('Exited', axis=1)
y = df['Exited']
print(X.shape)
print(X.columns.tolist())

(10000, 11)
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain']


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Verificar que es el MISMO test de la Fase 3
y_test_anterior = pd.read_csv('../data/processed/y_test.csv')['Exited']
assert (y_test.values == y_test_anterior.values).all(), "El test no coincide"
print("✅ Mismo test que en la Fase 3")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

✅ Mismo test que en la Fase 3
Train: (8000, 11) | Test: (2000, 11)


In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

base = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1,
                            subsample=0.8, colsample_bytree=0.8,
                            random_state=42, eval_metric='auc'))
])

scores = cross_val_score(base, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"AUC por fold: {scores.round(4)}")
print(f"AUC promedio (CV): {scores.mean():.4f} ± {scores.std():.4f}")

AUC por fold: [0.8375 0.8573 0.8519 0.8104 0.8292]
AUC promedio (CV): 0.8373 ± 0.0168


In [4]:
param_dist = {
    'model__n_estimators': randint(100, 600),
    'model__max_depth': randint(2, 7),
    'model__learning_rate': loguniform(0.01, 0.2),
    'model__subsample': uniform(0.6, 0.4),
    'model__colsample_bytree': uniform(0.6, 0.4),
    'model__min_child_weight': randint(1, 10),
    'model__gamma': uniform(0, 5),
    'model__reg_lambda': loguniform(0.1, 10),
}

search = RandomizedSearchCV(
    base, param_dist, n_iter=40, scoring='roc_auc', cv=cv,
    n_jobs=-1, random_state=42, return_train_score=True, verbose=1
)
search.fit(X_train, y_train)

print(f"\nMejor AUC (CV): {search.best_score_:.4f}")
print("Mejores hiperparámetros:")
for k, v in search.best_params_.items():
    print(f"  {k.replace('model__', '')}: {v}")

Fitting 5 folds for each of 40 candidates, totalling 200 fits

Mejor AUC (CV): 0.8414
Mejores hiperparámetros:
  colsample_bytree: 0.602208846849441
  gamma: 4.0773071422741705
  learning_rate: 0.08310795711416077
  max_depth: 4
  min_child_weight: 3
  n_estimators: 388
  reg_lambda: 1.6289957436723506
  subsample: 0.9705203514053395


In [5]:
res = pd.DataFrame(search.cv_results_)
best = res.loc[search.best_index_]
print(f"AUC train (promedio folds): {best['mean_train_score']:.4f}")
print(f"AUC validación (CV):        {best['mean_test_score']:.4f}")
print(f"Brecha:                     {best['mean_train_score'] - best['mean_test_score']:.4f}")

AUC train (promedio folds): 0.8705
AUC validación (CV):        0.8414
Brecha:                     0.0291


In [6]:
best_params = {k.replace('model__', ''): v for k, v in search.best_params_.items()}
ratio = (y_train == 0).sum() / (y_train == 1).sum()

sin_smote = XGBClassifier(**best_params, scale_pos_weight=ratio,
                          random_state=42, eval_metric='auc')
scores_ss = cross_val_score(sin_smote, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
print(f"Con SMOTE: {search.best_score_:.4f}")
print(f"Sin SMOTE (scale_pos_weight): {scores_ss.mean():.4f}")

Con SMOTE: 0.8414
Sin SMOTE (scale_pos_weight): 0.8660


In [7]:
final = search.best_estimator_
test_auc = roc_auc_score(y_test, final.predict_proba(X_test)[:, 1])

print(f"Test AUC (modelo con tuning): {test_auc:.4f}")
print(f"Test AUC (Fase 3, sin tuning): 0.8595")

Test AUC (modelo con tuning): 0.8523
Test AUC (Fase 3, sin tuning): 0.8595


In [9]:
from sklearn.model_selection import cross_validate

final = XGBClassifier(**best_params, scale_pos_weight=ratio,
                      random_state=42, eval_metric='auc')

# Brecha de sobreajuste del modelo elegido
cv_res = cross_validate(final, X_train, y_train, cv=cv, scoring='roc_auc',
                        return_train_score=True, n_jobs=-1)
print(f"AUC train (CV):      {cv_res['train_score'].mean():.4f}")
print(f"AUC validación (CV): {cv_res['test_score'].mean():.4f}")

# Entrenar con todo el train y evaluar UNA vez en test
final.fit(X_train, y_train)
test_auc = roc_auc_score(y_test, final.predict_proba(X_test)[:, 1])
print(f"\nTest AUC (modelo final, sin SMOTE): {test_auc:.4f}")

joblib.dump(final, '../models/xgboost_tuned.pkl')
X_test.to_csv('../data/processed/X_test_tuned.csv', index=False)
print("✅ Modelo final guardado")

AUC train (CV):      0.9041
AUC validación (CV): 0.8660

Test AUC (modelo final, sin SMOTE): 0.8713
✅ Modelo final guardado
